# YouTube Audio Download Pipeline

**Goal:** Download audio files as .wav for matched YouTube videos

**Input:** `youtube_matches.csv` (from matching notebook)

**Output:** `.wav` files in `data/raw/youtube_audio/` + download status CSV

---

## Pipeline Flow

1. Load YouTube matches
2. Filter to successfully matched songs
3. Download audio using yt-dlp (converts to WAV)
4. Save with `track_id` as filename
5. Track download status (success/failed/skipped)
6. Resume capability for interrupted runs

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import subprocess
import time
import os

## 2. Configuration

In [2]:
# Paths
MATCHES_PATH = "../data/raw/youtube_matches.csv"
AUDIO_OUTPUT_DIR = "../data/raw/youtube_audio/"
DOWNLOAD_STATUS_PATH = "../data/raw/youtube_download_status.csv"

# Create audio directory if it doesn't exist
Path(AUDIO_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Download settings
BATCH_SIZE = 50  # Save progress every 50 downloads
SLEEP_BETWEEN_DOWNLOADS = 1.0  # Seconds to wait between downloads

# Audio quality settings
AUDIO_FORMAT = 'wav'
AUDIO_QUALITY = '192'  # kbps for intermediate format before WAV conversion

print(f"Audio will be saved to: {AUDIO_OUTPUT_DIR}")
print(f"Download status will be saved to: {DOWNLOAD_STATUS_PATH}")

Audio will be saved to: ../data/raw/youtube_audio/
Download status will be saved to: ../data/raw/youtube_download_status.csv


## 3. Load YouTube Matches

In [3]:
# Load matches
matches_df = pd.read_csv(MATCHES_PATH)

print(f"Total matches loaded: {len(matches_df):,}")
print(f"Successfully matched: {matches_df['video_found'].sum():,}")
print(f"Not found: {(~matches_df['video_found']).sum():,}")

# Filter to only successfully matched songs
valid_matches = matches_df[matches_df['video_found']].copy()
print(f"\nSongs ready for download: {len(valid_matches):,}")

valid_matches.head()

Total matches loaded: 3,500
Successfully matched: 3,448
Not found: 52

Songs ready for download: 3,448


,track_id,track_name,artists,album_name,popularity,video_found,youtube_video_id,youtube_url,youtube_title,youtube_channel,youtube_duration,youtube_views,match_confidence,content_type
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,Gen Hoshino,Comedy,73,True,D_Oyplmhhv0,https://www.youtube.com/watch?v=D_Oyplmhhv0,星野源 – 喜劇 (Official Video),星野源 Gen Hoshino,275,36407179,0.48,music_video
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ben Woodward,Ghost (Acoustic),55,True,lcOJS28FTbY,https://www.youtube.com/watch?v=lcOJS28FTbY,Ben Woodward - Original,Ben Woodward,213,8567,0.60,music_video
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,Ingrid Michaelson;ZAYN,To Begin Again,57,True,gbh5JEOrwt0,https://www.youtube.com/watch?v=gbh5JEOrwt0,"Ingrid Michaelson, ZAYN - To Begin Again (Offi...",Fox Music,210,20,0.72,music_video
3,6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,71,True,tyoAsprwDRc,https://www.youtube.com/watch?v=tyoAsprwDRc,Kina Grannis - Can't Help Falling In Love (Fro...,Kina Grannis,330,18237684,1.00,music_video
4,5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet,Hold On,82,True,a0sxc2jIYD0,https://www.youtube.com/watch?v=a0sxc2jIYD0,Chord Overstreet - Hold On (Audio),Chord Overstreet,201,3216606,1.00,official_audio


## 4. Download Functions

In [4]:
def download_audio(youtube_url, output_path, track_id):
    """
    Download audio from YouTube URL and save as WAV file
    
    Returns:
        dict with status, file_path, error_message, file_size_mb
    """
    try:
        # Build yt-dlp command
        cmd = [
            'yt-dlp',
            '--extract-audio',
            '--audio-format', 'wav',
            '--audio-quality', '0',  # Best quality
            '--output', output_path,
            '--no-playlist',
            '--no-warnings',
            '--quiet',
            '--no-progress',
            youtube_url
        ]
        
        # Run download with timeout
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=300  # 5 minute timeout per song
        )
        
        # Check if file was created
        if Path(output_path).exists():
            file_size_mb = Path(output_path).stat().st_size / (1024 * 1024)
            return {
                'status': 'success',
                'file_path': output_path,
                'error_message': '',
                'file_size_mb': round(file_size_mb, 2)
            }
        else:
            return {
                'status': 'failed',
                'file_path': '',
                'error_message': 'File not created after download',
                'file_size_mb': 0
            }
            
    except subprocess.TimeoutExpired:
        return {
            'status': 'failed',
            'file_path': '',
            'error_message': 'Download timeout (>5 minutes)',
            'file_size_mb': 0
        }
    except Exception as e:
        return {
            'status': 'failed',
            'file_path': '',
            'error_message': str(e),
            'file_size_mb': 0
        }

## 5. Test Single Download

In [5]:
# Test with first valid match
test_song = valid_matches.iloc[0]

print(f"Testing download for:")
print(f"  Track: {test_song['track_name']}")
print(f"  Artist: {test_song['artists']}")
print(f"  URL: {test_song['youtube_url']}")
print(f"\nDownloading...\n")

test_output_path = f"{AUDIO_OUTPUT_DIR}{test_song['track_id']}.wav"
test_result = download_audio(
    test_song['youtube_url'],
    test_output_path,
    test_song['track_id']
)

print(f"Result: {test_result['status']}")
if test_result['status'] == 'success':
    print(f"  File: {test_result['file_path']}")
    print(f"  Size: {test_result['file_size_mb']} MB")
else:
    print(f"  Error: {test_result['error_message']}")

Testing download for:
  Track: Comedy
  Artist: Gen Hoshino
  URL: https://www.youtube.com/watch?v=D_Oyplmhhv0

Downloading...

Result: success
  File: ../data/raw/youtube_audio/5SuOikwiRyPMVoIQDJUgSV.wav
  Size: 50.41 MB


## 6. Batch Download with Resume

In [18]:
# Check for existing downloads and status file
status_path = Path(DOWNLOAD_STATUS_PATH)

if status_path.exists():
    print("Found existing download status file")
    download_status = pd.read_csv(DOWNLOAD_STATUS_PATH)
    print(f"Already processed: {len(download_status):,} songs")
    print(f"  Successful: {(download_status['download_status'] == 'success').sum():,}")
    print(f"  Failed: {(download_status['download_status'] == 'failed').sum():,}")
    print(f"  Skipped: {(download_status['download_status'] == 'skipped').sum():,}")
    
    # Find songs that still need processing
    processed_ids = set(download_status['track_id'].values)
    pending_downloads = valid_matches[~valid_matches['track_id'].isin(processed_ids)].copy()
    print(f"\nRemaining to download: {len(pending_downloads):,} songs")
else:
    print("No existing downloads found. Starting fresh.")
    download_status = None
    pending_downloads = valid_matches.copy()
    print(f"Total to download: {len(pending_downloads):,} songs")

Found existing download status file
Already processed: 1,200 songs
  Successful: 1,197
  Failed: 0
  Skipped: 3

Remaining to download: 2,248 songs


In [15]:
def process_downloads(df, batch_size=50):
    """
    Download audio files in batches with progress saving
    """
    all_status = []
    total_songs = len(df)
    
    # Load existing status if available
    if Path(DOWNLOAD_STATUS_PATH).exists():
        existing = pd.read_csv(DOWNLOAD_STATUS_PATH)
        all_status = existing.to_dict('records')
        print(f"Loaded {len(all_status):,} existing records\n")
    
    # Process in batches
    for batch_start in range(0, total_songs, batch_size):
        batch_end = min(batch_start + batch_size, total_songs)
        batch_df = df.iloc[batch_start:batch_end]
        
        print(f"\n{'='*60}")
        print(f"Processing batch: {batch_start + 1}-{batch_end} of {total_songs:,}")
        print(f"{'='*60}\n")
        
        batch_status = []
        
        for idx, row in tqdm(batch_df.iterrows(), total=len(batch_df), desc=f"Batch {batch_start//batch_size + 1}"):
            output_path = f"{AUDIO_OUTPUT_DIR}{row['track_id']}.wav"
            
            # Check if file already exists
            if Path(output_path).exists():
                file_size_mb = Path(output_path).stat().st_size / (1024 * 1024)
                result = {
                    'status': 'skipped',
                    'file_path': output_path,
                    'error_message': 'File already exists',
                    'file_size_mb': round(file_size_mb, 2)
                }
            else:
                # Download the file
                result = download_audio(
                    row['youtube_url'],
                    output_path,
                    row['track_id']
                )
            
            # Record status
            status_record = {
                'track_id': row['track_id'],
                'track_name': row['track_name'],
                'artists': row['artists'],
                'youtube_video_id': row['youtube_video_id'],
                'youtube_url': row['youtube_url'],
                'download_status': result['status'],
                'file_path': result['file_path'],
                'file_size_mb': result['file_size_mb'],
                'error_message': result['error_message']
            }
            
            batch_status.append(status_record)
            
            # Sleep between downloads (be nice to YouTube)
            if result['status'] != 'skipped':
                time.sleep(SLEEP_BETWEEN_DOWNLOADS)
        
        # Add batch results to all status
        all_status.extend(batch_status)
        
        # Save progress after each batch
        status_df = pd.DataFrame(all_status)
        status_df.to_csv(DOWNLOAD_STATUS_PATH, index=False)
        
        print(f"\n✓ Batch complete. Saved progress: {len(all_status):,} total records")
        print(f"  Successful in this batch: {sum(s['download_status'] == 'success' for s in batch_status)}")
        print(f"  Failed in this batch: {sum(s['download_status'] == 'failed' for s in batch_status)}")
        print(f"  Skipped in this batch: {sum(s['download_status'] == 'skipped' for s in batch_status)}")
    
    return pd.DataFrame(all_status)

# Set limit for testing (comment out to process all)
TEST_LIMIT = None  # Start with 100 songs to test

if TEST_LIMIT:
    pending_limited = pending_downloads.head(TEST_LIMIT)
    print(f"\n⚠️  TEST MODE: Processing only {TEST_LIMIT:,} songs")
    print(f"To process all songs, set TEST_LIMIT = None\n")
else:
    pending_limited = pending_downloads
    print(f"\n▶ FULL MODE: Processing all {len(pending_downloads):,} songs\n")


▶ FULL MODE: Processing all 2,548 songs



In [16]:
# Run the download process
# Uncomment the line below when ready:

download_results = process_downloads(pending_limited, batch_size=BATCH_SIZE)

Loaded 900 existing records


Processing batch: 1-50 of 2,548



Batch 1:   0%|          | 0/50 [00:00<?, ?it/s]


✓ Batch complete. Saved progress: 950 total records
  Successful in this batch: 48
  Failed in this batch: 0
  Skipped in this batch: 2

Processing batch: 51-100 of 2,548



Batch 2:   0%|          | 0/50 [00:00<?, ?it/s]


✓ Batch complete. Saved progress: 1,000 total records
  Successful in this batch: 50
  Failed in this batch: 0
  Skipped in this batch: 0

Processing batch: 101-150 of 2,548



...


✓ Batch complete. Saved progress: 1,150 total records
  Successful in this batch: 50
  Failed in this batch: 0
  Skipped in this batch: 0

Processing batch: 251-300 of 2,548



Batch 6:   0%|          | 0/50 [00:00<?, ?it/s]


✓ Batch complete. Saved progress: 1,200 total records
  Successful in this batch: 50
  Failed in this batch: 0
  Skipped in this batch: 0

Processing batch: 301-350 of 2,548



Batch 7:   0%|          | 0/50 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 7. Analyze Download Results

In [17]:
# Load download status
if Path(DOWNLOAD_STATUS_PATH).exists():
    status_df = pd.read_csv(DOWNLOAD_STATUS_PATH)
    
    print("=== Download Results ===")
    print(f"\nTotal processed: {len(status_df):,}")
    print(f"Successful: {(status_df['download_status'] == 'success').sum():,} ({(status_df['download_status'] == 'success').mean():.1%})")
    print(f"Failed: {(status_df['download_status'] == 'failed').sum():,}")
    print(f"Skipped (already exist): {(status_df['download_status'] == 'skipped').sum():,}")
    
    # File size stats
    successful = status_df[status_df['download_status'] == 'success']
    if len(successful) > 0:
        print(f"\n=== File Size Stats ===")
        print(f"Total size: {successful['file_size_mb'].sum():.2f} MB")
        print(f"Average size: {successful['file_size_mb'].mean():.2f} MB per file")
        print(f"Min size: {successful['file_size_mb'].min():.2f} MB")
        print(f"Max size: {successful['file_size_mb'].max():.2f} MB")
    
    # Error analysis
    failed = status_df[status_df['download_status'] == 'failed']
    if len(failed) > 0:
        print(f"\n=== Common Errors ===")
        error_counts = failed['error_message'].value_counts().head(5)
        for error, count in error_counts.items():
            print(f"  {error}: {count} times")
    
    # Show sample results
    print(f"\n=== Sample Successful Downloads ===")
    display(successful[["track_name", "artists", "file_size_mb", "file_path"]].head(10))
else:
    print("No download status file found yet. Run the download process first.")

=== Download Results ===

Total processed: 1,200
Successful: 1,197 (99.8%)
Failed: 0
Skipped (already exist): 3

=== File Size Stats ===
Total size: 54228.97 MB
Average size: 45.30 MB per file
Min size: 3.02 MB
Max size: 839.22 MB

=== Sample Successful Downloads ===


,track_name,artists,file_size_mb,file_path
1,Ghost - Acoustic,Ben Woodward,38.96,../data/raw/youtube_audio/4qPNDBW1i3p13qLCt0Ki...
2,To Begin Again,Ingrid Michaelson;ZAYN,38.39,../data/raw/youtube_audio/1iJBSr7s7jYXzM8EGcbK...
3,Can't Help Falling In Love,Kina Grannis,60.46,../data/raw/youtube_audio/6lfxq3CG4xtTiEg7opyC...
4,Hold On,Chord Overstreet,36.87,../data/raw/youtube_audio/5vjLSffimiIP26QG5WcN...
5,Days I Will Remember,Tyrone Wells,39.23,../data/raw/youtube_audio/01MVOl9KtVTNfFiBU9I7...
6,Say Something,A Great Big World;Christina Aguilera,42.35,../data/raw/youtube_audio/6Vc5wAMmXdKIAM7WUoEb...
7,I'm Yours,Jason Mraz,40.46,../data/raw/youtube_audio/1EzrEOXmMH3G43AXT1y7...
8,Lucky,Jason Mraz;Colbie Caillat,37.04,../data/raw/youtube_audio/0IktbUcnAGrvD03AWnz3...
9,Hunger,Ross Copperman,37.65,../data/raw/youtube_audio/7k9GuJYLp2AzqokyEdwE...
10,Give Me Your Forever,Zack Tabudlo,66.89,../data/raw/youtube_audio/4mzP5mHkRvGxdhdGdAH7...


## 8. Next Steps

✓ **You now have:** `.wav` audio files in `data/raw/youtube_audio/`

**Next notebook:** `04c_youtube_audio_features.ipynb`
- Extract librosa features from downloaded audio
- Your existing feature extraction pipeline
- Save to final features dataset